In [1]:
import SLiCAP as sl
import os
sl.initProject("NMOS lookup", notebook=True)
netlist = sl.makeCircuit("kicad/CSstage/NMOS.kicad_sch", language="SPICE")
netlistlines = netlist.splitlines()[1:-1]
netlist = "\n".join(netlistlines)

Compiling library: SLiCAP.lib.
Compiling library: SLiCAPmodels.lib.
Creating netlist of kicad/CSstage/NMOS.kicad_sch using KiCAD
Creating drawing-size SVG and PDF images of kicad/CSstage/NMOS.kicad_sch
Plotted to '/home/anton/DATA/www/SEDwebsite/_build/SEDwebsite/EE4109-2025-2026/designExample/Technology/CSstage/CSstageSLiCAP/img/NMOS.svg'.
Done.


In [2]:
f = open("txt/control.txt", "r")
control = f.read()
f.close()

In [3]:
params = {"W": 220e-9,
          "L": 180e-9,
          "M": 1,
          "ID": 10e-6,
          "VS": 0,
          "VB": 0,
          "VD": 0.9,
          "F": 1e9}

In [4]:
simfile = control
for key in params.keys():
    simfile = simfile.replace("<" + key + ">", str(params[key]))
simfile = simfile.replace("<NETLIST>", netlist)
f = open("cir/NMOS_lookup.cir", "w")
f.write(simfile)
f.close()

In [5]:
os.system(sl.ini.ngspice + " -b cir/NMOS_lookup.cir")


Note: No compatibility mode selected!


Circuit: mos_lookup

Doing analysis at TEMP = 27.000000 and TNOM = 27.000000


No. of Data Rows : 1
Doing analysis at TEMP = 27.000000 and TNOM = 27.000000


No. of Data Rows : 1
Doing analysis at TEMP = 27.000000 and TNOM = 27.000000


No. of Data Rows : 1
Doing analysis at TEMP = 27.000000 and TNOM = 27.000000


No. of Data Rows : 1
ASCII raw file "MOS_OP.out"
Note: Simulation executed from .control section 


Using SPARSE 1.3 as Direct Linear Solver

Note: Starting dynamic gmin stepping
Note: Dynamic gmin stepping completed
Using SPARSE 1.3 as Direct Linear Solver

Note: Starting dynamic gmin stepping
Note: Dynamic gmin stepping completed
Using SPARSE 1.3 as Direct Linear Solver

Note: Starting dynamic gmin stepping
Note: Dynamic gmin stepping completed
Using SPARSE 1.3 as Direct Linear Solver

Note: Starting dynamic gmin stepping
Note: Dynamic gmin stepping completed


0

In [6]:
f = open("MOS_OP.out", "r")
lines = f.readlines()
f.close()
Variables = False
Values    = False
names     = []
output    = {}
for l in lines:
    if l == "Variables:\n":
        Variables = True
    elif l == "Values:\n":
        Variables = False
        Values    = True
    elif Variables:
        i, name, var_type = l[0:-1].split()
        names.append(name)
        i = 0
    elif Values:
        if l[0] == "0":
            n, value = l[:-1].split()
        elif l != "\n":
            if i != 0:
                output[names[i]] = float(l.strip())
            i += 1

In [7]:
model = {}
model["c_gs"] = 0.5*(output["cgs"] + output["csg"])
model["c_gd"] = 0.5*(output["cgd"] + output["cdg"])
model["c_gb"] = 0.5*(output["cgb"] + output["cbg"])
model["c_ds"] = 0.5*(output["cds"] + output["csd"])
model["c_db"] = 0.5*(output["cdb"] + output["cbd"])
model["c_sb"] = 0.5*(output["cbs"] + output["csb"])
model["g_m"]  = 0.5*(output["ggs"] - output["ggd"])
model["g_o"]  = output["gds"]
model["g_b"]  = 0.5*(output["gbs"] - output["gbd"])
model["v_gs"] = output["v(vgs)"]
model["v_ds"] = output["v(vds)"]
model["i_ds"] = output["i(ids)"]

In [8]:
sl.backAnnotateSchematic("kicad/CSstage/NMOS.kicad_sch", model)

Creating drawing-size SVG and PDF images of kicad/CSstage/NMOS.kicad_sch
Plotted to '/home/anton/DATA/www/SEDwebsite/_build/SEDwebsite/EE4109-2025-2026/designExample/Technology/CSstage/CSstageSLiCAP/img/NMOS.svg'.
Done.


In [9]:
sl.img2html("NMOS.svg",600)

In [10]:
for param in model.keys():
    print(param, "\t:", "{:+e}".format(model[param]))

c_gs 	: +2.275502e-16
c_gd 	: +7.339472e-17
c_gb 	: +6.478905e-17
c_ds 	: -2.018740e-19
c_db 	: +2.935420e-16
c_sb 	: +4.181389e-16
g_m 	: +8.476809e-05
g_o 	: +2.566975e-06
g_b 	: +2.296353e-05
v_gs 	: +6.486426e-01
v_ds 	: +8.998764e-01
i_ds 	: +9.999998e-06


In [11]:
%store model

Stored 'model' (dict)
